# pandas I: Series, DataFrames, Input/Output, Selection and Data Types

Notebook for **Chapter 4** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch04.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
import pandas as pd
import numpy as np

## 4.1 Series: One Labelled Column

In [ ]:
s = pd.Series([120, 95, 310, 88],
              index=["delhi", "pune", "leh", "goa"],
              name="hours")
print(s)
print(s["leh"], s.iloc[0], s.mean())
print(s[s > 100])

In [ ]:
print(s + pd.Series({"delhi": 1, "pune": 2}))

delhi    121.0
goa        NaN
leh        NaN
pune      97.0
dtype: float64


## 4.2 DataFrame: A Table of Series

In [ ]:
import io
csv = """order_id,order_date,customer_id,city,qty,price,status
1001,2025-01-03,C01,Pune,2,499.0,delivered
1002,2025-01-03,C02,Mumbai,1,1299.5,delivered
1003,2025-01-04,C01,Pune,3,199.0,returned
1004,2025-01-05,C03,Delhi,1,,delivered
1005,2025-01-06,C04,Mumbai,5,89.9,cancelled
1006,2025-01-07,C02,Mumbai,2,649.0,delivered
"""
df = pd.read_csv(io.StringIO(csv), parse_dates=["order_date"],
                 dtype={"customer_id": "string"})
print(df)
print(df.dtypes)
print(df.shape)
df.info()

In [ ]:
df["amount"] = df["qty"] * df["price"]
print(df.nlargest(2, "amount")[["order_id", "amount"]])

## 4.3 Reading and Writing Files

In [ ]:
df.to_csv("orders_small.csv", index=False)
print(pd.read_csv("orders_small.csv", usecols=["order_id", "price"], nrows=3))

   order_id   price
0      1001   499.0
1      1002  1299.5
2      1003   199.0


In [ ]:
for chunk in pd.read_csv("orders_small.csv", chunksize=4):
    print(len(chunk), chunk.qty.sum())

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
df.drop(columns=["status"]).assign(order_date=df.order_date.astype(str)).to_sql("orders", con, index=False)
print(pd.read_sql("select city, sum(qty*price) as revenue from orders group by city", con))

     city  revenue
0   Delhi      NaN
1  Mumbai   3047.0
2    Pune   1595.0


## 4.4 Selecting Rows and Columns

In [ ]:
print(df.loc[df.city == "Mumbai", ["order_id", "price"]])
print(df.iloc[1:4, 0:3])

   order_id   price
1      1002  1299.5
4      1005    89.9
5      1006   649.0
   order_id order_date customer_id
1      1002 2025-01-03         C02
2      1003 2025-01-04         C01
3      1004 2025-01-05         C03


In [ ]:
print(df.query("qty >= 2 and status != 'cancelled'"))

   order_id order_date customer_id    city  qty  price     status
0      1001 2025-01-03         C01    Pune    2  499.0  delivered
2      1003 2025-01-04         C01    Pune    3  199.0   returned
5      1006 2025-01-07         C02  Mumbai    2  649.0  delivered


## 4.5 Data Types and Common Pitfalls

In [ ]:
print(pd.Series([1, 2, None]).dtype)
print(pd.Series([1, 2, None], dtype="Int64").tolist(), pd.Series([1, 2, None], dtype="Int64").dtype)

float64
[1, 2, <NA>] Int64


In [ ]:
pd.to_numeric(pd.Series(["10", "20", "x"]), errors="coerce").tolist()   # [10.0, 20.0, nan]
pd.Series(["10", "20", "x"]).astype(int)    # ValueError: invalid literal for int() with base 10: 'x'
pd.to_datetime("03/04/2025")                # 2025-03-04 (month first)
pd.to_datetime("03/04/2025", dayfirst=True) # 2025-04-03 (day first)

In [ ]:
view = df[df.city == "Pune"]
view["qty"] = 0
print(df.qty.tolist())            # [2, 1, 3, 1, 5, 2]  (original unchanged)

df2 = df.copy()
df2[df2.city == "Pune"]["qty"] = 99      # chained assignment
print(df2.qty.tolist())           # [2, 1, 3, 1, 5, 2]  (nothing happened!)
df2.loc[df2.city == "Pune", "qty"] = 99  # correct: one step
print(df2.qty.tolist())           # [99, 1, 99, 1, 5, 2]

## 4.6 MultiIndex Deep Dive: Rows and Columns with Several Levels

In [ ]:
import pandas as pd
sales = pd.DataFrame({
    "city": ["Pune", "Pune", "Pune", "Pune", "Delhi", "Delhi", "Delhi", "Delhi"],
    "month": ["jan", "jan", "feb", "feb", "jan", "jan", "feb", "feb"],
    "product": ["tea", "coffee"] * 4,
    "revenue": [200, 150, 260, 120, 300, 90, 280, 110]})
mi = sales.set_index(["city", "month", "product"]).sort_index()
print(mi.loc["Pune"])                                # 1. first level only
print(mi.loc[("Pune", "feb")])                        # 2. first two levels
print(mi.loc[("Pune", "feb", "tea"), "revenue"])      # 3. a single cell
print(mi.xs("tea", level="product"))                  # 4. one value of the LAST level
print(mi.loc[(slice(None), "jan"), :])                # 5. every city, but only January

               revenue
month product         
feb   coffee       120
      tea          260
jan   coffee       150
      tea          200
         revenue
product         
coffee       120
tea          260
260
             revenue
city  month         
Delhi feb        280
      jan        300
Pune  feb        260
      jan        200
                     revenue
city  month product         
Delhi jan   coffee        90
            tea          300
Pune  jan   coffee       150
            tea          200


In [ ]:
import pandas as pd
sales = pd.DataFrame({
    "city": ["Pune", "Pune", "Pune", "Pune", "Delhi", "Delhi", "Delhi", "Delhi"],
    "month": ["jan", "jan", "feb", "feb", "jan", "jan", "feb", "feb"],
    "product": ["tea", "coffee"] * 4,
    "revenue": [200, 150, 260, 120, 300, 90, 280, 110]})
mi = sales.set_index(["city", "month", "product"]).sort_index()
print(mi.groupby(level=["city", "month"]).sum())          # drop the product level
print(mi["revenue"].unstack("month"))                     # months become columns
print(mi["revenue"].unstack("product").sum(axis=1))       # total per city and month
print(mi.reset_index().head(2))                           # back to a flat table

             revenue
city  month         
Delhi feb        390
      jan        390
Pune  feb        380
      jan        350
month          feb  jan
city  product          
Delhi coffee   110   90
      tea      280  300
Pune  coffee   120  150
      tea      260  200
city   month
Delhi  feb      390
       jan      390
Pune   feb      380
       jan      350
dtype: int64
    city month product  revenue
0  Delhi   feb  coffee      110
1  Delhi   feb     tea      280


In [ ]:
import pandas as pd
cols = pd.MultiIndex.from_product([["jan", "feb"], ["rev", "qty"]])
w = pd.DataFrame([[200, 10, 260, 12], [300, 14, 280, 13]], index=["Pune", "Delhi"], columns=cols)
print(w)
print(w["jan"])                      # all columns under jan
print(w[("feb", "rev")])             # one column
w.columns = ["_".join(c) for c in w.columns]    # flatten the two header rows into one
print(list(w.columns))

       jan      feb    
       rev qty  rev qty
Pune   200  10  260  12
Delhi  300  14  280  13
       rev  qty
Pune   200   10
Delhi  300   14
Pune     260
Delhi    280
Name: (feb, rev), dtype: int64
['jan_rev', 'jan_qty', 'feb_rev', 'feb_qty']


## Chapter recap

## Practice set

Problem 1 For `s = pd.Series([5, 7, 9], index=["a", "b", "c"])`, what is `s.loc["b"]` and `s.iloc[2]`?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch04.html